In [1]:
import pandas as pd
import re

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/My Drive/CADE

/content/drive/My Drive/CADE


## Classification

In [4]:
import pandas as pd
import re

df = pd.read_csv('/content/drive/My Drive/CADE/results/deepseekv3.2/results_classification.csv')

# Known labels across the dataset
LABELS = ['walking', 'jogging', 'sitting', 'standing', 'upstairs', 'downstairs',
          'no freeze', 'freeze', 'normal', 'abnormal', 'positive', 'negative',
          'supraventricular', 'premature ventricular', 'fusion', 'unclassifiable',
          'atrial premature', 'premature ventricular contraction', 'right bundle branch block',
          'left bundle branch block', 'paced']
# extract the last label of the sentence
def extract_label(text):
    text = str(text).lower() # converts everything to lowercase
    # Find the last occurrence of a known label
    best = None
    best_pos = -1
    for label in sorted(LABELS, key=len, reverse=True):  # longest first, prevents "ventricular" from being caught when the full label is actually "premature ventricular."
        pos = text.rfind(label) # rfind(): returns the index where is string is found. If not found, it returns -1
        if pos != -1 and pos > best_pos: # If a label is found later in the text than any previously found label, the script "overwrites" the old one
            best_pos = pos
            best = label
    return best

correct = 0
total = len(df)
mismatches = []
for i, row in df.iterrows():
    qa = str(row['QA_list'])
    # Extract answer portion
    ans_match = re.search(r'answer.*?[:,]\s*["\s]*(.*?)["\s]*$', qa, re.IGNORECASE | re.DOTALL) # regular expression
    true_text = ans_match.group(1) if ans_match else qa

    true_label = extract_label(true_text)
    model_label = extract_label(str(row['model_response']))

    if true_label and model_label and true_label == model_label:
        correct += 1
    else:
        if len(mismatches) < 5:
            mismatches.append((i, true_label, model_label, str(row['model_response'])[:100]))

print(f"Correct: {correct}/{total}")
print(f"Accuracy: {correct/total*100:.2f}%")
print(f"\nSample mismatches:")
for idx, t, m, resp in mismatches:
    print(f"  Row {idx}: true='{t}' vs model='{m}'")

# Check for any None labels
none_true = sum(1 for _, row in df.iterrows() if extract_label(str(row['QA_list'])) is None)
none_model = sum(1 for _, row in df.iterrows() if extract_label(str(row['model_response'])) is None)
print(f"\nUnparsed true labels: {none_true}, Unparsed model labels: {none_model}")
if none_model > 0:
    for i, row in df.iterrows():
        if extract_label(str(row['model_response'])) is None:
            print(f"  Row {i} model_response: {str(row['model_response'])[:200]}")

Correct: 291/400
Accuracy: 72.75%

Sample mismatches:
  Row 0: true='sitting' vs model='standing'
  Row 4: true='upstairs' vs model='walking'
  Row 8: true='jogging' vs model='walking'
  Row 9: true='jogging' vs model='walking'
  Row 13: true='sitting' vs model='standing'

Unparsed true labels: 0, Unparsed model labels: 0


## Anomaly Detection

In [5]:
import pandas as pd
import re

df = pd.read_csv('/content/drive/My Drive/CADE/results/deepseekv3.2/results_anomaly.csv')
print(f"Total rows: {len(df)}")

def extract_label(text):
    """Extract anomaly detection label: 'normal' or 'anomaly'."""
    text = str(text).lower()
    if 'normal' in text or 'normal point' in text or 'no anomalies' in text:
        return 'normal'
    if 'anomaly point' in text or 'anomaly' in text or 'anomalies' in text:
        return 'anomaly'
    return None

correct = 0
total = len(df)
mismatches = []
for i, row in df.iterrows():
    qa = str(row['QA_list'])
    # Extract answer portion from QA_list
    ans_match = re.search(r'answer.*?[:,]\s*["\s]*(.*?)["\s]*$', qa, re.IGNORECASE | re.DOTALL)
    true_text = ans_match.group(1) if ans_match else qa

    true_label = extract_label(true_text)
    model_label = extract_label(str(row['model_response']))

    if true_label and model_label and true_label == model_label:
        correct += 1
    else:
        mismatches.append((i, true_label, model_label))

print(f"Correct: {correct}/{total}")
print(f"Accuracy: {correct/total*100:.2f}%")
print(f"\nMismatches ({len(mismatches)}):")
for idx, t, m in mismatches:
    print(f"  Row {idx}: true='{t}' vs model='{m}'")

Total rows: 400
Correct: 239/400
Accuracy: 59.75%

Mismatches (161):
  Row 3: true='normal' vs model='anomaly'
  Row 4: true='anomaly' vs model='normal'
  Row 6: true='anomaly' vs model='normal'
  Row 7: true='anomaly' vs model='normal'
  Row 9: true='normal' vs model='anomaly'
  Row 10: true='normal' vs model='anomaly'
  Row 15: true='normal' vs model='anomaly'
  Row 16: true='normal' vs model='anomaly'
  Row 17: true='anomaly' vs model='normal'
  Row 19: true='anomaly' vs model='normal'
  Row 20: true='normal' vs model='anomaly'
  Row 21: true='normal' vs model='anomaly'
  Row 23: true='normal' vs model='anomaly'
  Row 24: true='anomaly' vs model='normal'
  Row 25: true='normal' vs model='anomaly'
  Row 28: true='normal' vs model='anomaly'
  Row 29: true='normal' vs model='anomaly'
  Row 31: true='anomaly' vs model='normal'
  Row 39: true='normal' vs model='anomaly'
  Row 40: true='normal' vs model='anomaly'
  Row 43: true='normal' vs model='anomaly'
  Row 47: true='anomaly' vs model

## Multiple Choice

In [6]:
import pandas as pd
import re

df = pd.read_csv('/content/drive/My Drive/CADE/results/deepseekv3.2/results_multiple_choice.csv')
print(f"Total rows: {len(df)}")

def extract_choice(text):
    """Extract multiple choice answer (A/B/C/D or True/False) from text."""
    text = str(text).strip()

    # First: extract content from <answer>...</answer> tags if present
    ans_tag = re.search(r'<answer>\s*(.*?)\s*</answer>', text, re.IGNORECASE | re.DOTALL)
    if ans_tag:
        text = ans_tag.group(1).strip()

    # Match a single letter A-D (possibly the entire text)
    if re.match(r'^[A-Da-d]$', text):
        return text.upper()

    # Match leading letter choice like "A)", "a)", "B.", "C)" etc.
    m = re.match(r'^([A-Da-d])[).\s]', text)
    if m:
        return m.group(1).upper()

    # "Answer: (b)" or "answer is (A)" or "option (c)"
    m = re.search(r'(?:answer|option)[:\s]+\(?([A-Da-d])\)?', text, re.IGNORECASE)
    if m:
        return m.group(1).upper()

    # "The answer is (A)"
    m = re.search(r'the answer is\s*\(?([A-Da-d])\)?', text, re.IGNORECASE)
    if m:
        return m.group(1).upper()

    # True/False style answers
    if re.match(r'^true', text, re.IGNORECASE):
        return 'TRUE'
    if re.match(r'^false', text, re.IGNORECASE):
        return 'FALSE'

    # Find standalone letter in parentheses: (A), (B), etc.
    m = re.search(r'\(([A-Da-d])\)', text)
    if m:
        return m.group(1).upper()

    # Look for bold letter like **A** or **B**
    m = re.search(r'\*\*([A-Da-d])\*\*', text)
    if m:
        return m.group(1).upper()

    # Look for "option X" or "choice X" pattern
    m = re.search(r'(?:option|choice)\s+([A-Da-d])\b', text, re.IGNORECASE)
    if m:
        return m.group(1).upper()

    # Look for bold text with answer like **B) Low volatility**
    m = re.search(r'\*\*([A-Da-d])[).]', text)
    if m:
        return m.group(1).upper()

    # Last resort: find first standalone A-D letter
    m = re.search(r'\b([A-Da-d])\)', text)
    if m:
        return m.group(1).upper()

    return None

correct = 0
total = len(df)
mismatches = []
unparsed_true = []
unparsed_model = []

for i, row in df.iterrows():
    qa = str(row['QA_list'])
    # Extract answer portion from QA_list
    ans_match = re.search(r'answer.*?[:,]\s*["\s]*(.*?)["\s]*$', qa, re.IGNORECASE | re.DOTALL)
    true_text = ans_match.group(1) if ans_match else qa
    true_label = extract_choice(true_text)

    model_text = str(row['model_response'])
    model_label = extract_choice(model_text)

    if true_label and model_label and true_label == model_label:
        correct += 1
    else:
        mismatches.append((i, true_label, model_label))

    if true_label is None:
        unparsed_true.append((i, true_text[:150]))
    if model_label is None:
        unparsed_model.append((i, model_text[:150]))

print(f"Correct: {correct}/{total}")
print(f"Accuracy: {correct/total*100:.2f}%")
print(f"\nMismatches ({len(mismatches)}):")
for idx, t, m in mismatches:
    print(f"  Row {idx}: true='{t}' vs model='{m}'")
if unparsed_true:
    print(f"\nUnparsed TRUE labels ({len(unparsed_true)}):")
    for idx, r in unparsed_true:
        print(f"  Row {idx}: {r}")
if unparsed_model:
    print(f"\nUnparsed MODEL responses ({len(unparsed_model)}):")
    for idx, r in unparsed_model:
        print(f"  Row {idx}: {r}")

Total rows: 400
Correct: 234/400
Accuracy: 58.50%

Mismatches (166):
  Row 4: true='C' vs model='B'
  Row 6: true='C' vs model='D'
  Row 7: true='C' vs model='A'
  Row 11: true='A' vs model='B'
  Row 15: true='B' vs model='None'
  Row 17: true='None' vs model='None'
  Row 18: true='B' vs model='A'
  Row 20: true='D' vs model='C'
  Row 21: true='A' vs model='None'
  Row 23: true='C' vs model='B'
  Row 24: true='B' vs model='D'
  Row 25: true='B' vs model='C'
  Row 26: true='A' vs model='B'
  Row 30: true='None' vs model='None'
  Row 31: true='A' vs model='B'
  Row 33: true='A' vs model='B'
  Row 34: true='A' vs model='B'
  Row 36: true='A' vs model='None'
  Row 37: true='B' vs model='A'
  Row 38: true='A' vs model='C'
  Row 40: true='B' vs model='A'
  Row 43: true='B' vs model='C'
  Row 45: true='A' vs model='D'
  Row 46: true='A' vs model='B'
  Row 48: true='A' vs model='B'
  Row 56: true='C' vs model='B'
  Row 57: true='A' vs model='D'
  Row 60: true='A' vs model='None'
  Row 63: true

## True/False Question

In [7]:
import pandas as pd
import re

df = pd.read_csv('/content/drive/My Drive/CADE/results/deepseekv3.2/results_true_false.csv')
print(f"Total rows: {len(df)}")

def extract_tf(text):
    """Extract True/False label from text."""
    text = str(text).strip().lower()
    # Check the leading word first
    if re.match(r'^(true|yes)', text):
        return 'true'
    if re.match(r'^(false|no[^a-z])', text):
        return 'false'
    # Fallback: search anywhere (only if unambiguous)
    if 'true' in text and 'false' not in text:
        return 'true'
    if 'false' in text and 'true' not in text:
        return 'false'
    return None

correct = 0
total = len(df)
mismatches = []
for i, row in df.iterrows():
    qa = str(row['QA_list'])
    # Extract answer portion from QA_list
    ans_match = re.search(r'answer.*?[:,]\s*["\s]*(.*?)["\s]*$', qa, re.IGNORECASE | re.DOTALL)
    true_text = ans_match.group(1) if ans_match else qa

    true_label = extract_tf(true_text)
    model_label = extract_tf(str(row['model_response']))

    if true_label and model_label and true_label == model_label:
        correct += 1
    else:
        mismatches.append((i, true_label, model_label))

print(f"Correct: {correct}/{total}")
print(f"Accuracy: {correct/total*100:.2f}%")
print(f"\nMismatches ({len(mismatches)}):")
for idx, t, m in mismatches:
    print(f"  Row {idx}: true='{t}' vs model='{m}'")

Total rows: 400
Correct: 299/400
Accuracy: 74.75%

Mismatches (101):
  Row 6: true='false' vs model='true'
  Row 8: true='true' vs model='false'
  Row 17: true='false' vs model='true'
  Row 19: true='false' vs model='true'
  Row 28: true='false' vs model='true'
  Row 29: true='false' vs model='true'
  Row 34: true='false' vs model='true'
  Row 35: true='false' vs model='true'
  Row 37: true='false' vs model='true'
  Row 39: true='false' vs model='true'
  Row 42: true='false' vs model='true'
  Row 47: true='false' vs model='true'
  Row 48: true='false' vs model='true'
  Row 50: true='false' vs model='true'
  Row 52: true='false' vs model='true'
  Row 55: true='true' vs model='false'
  Row 64: true='false' vs model='true'
  Row 67: true='true' vs model='false'
  Row 73: true='false' vs model='true'
  Row 84: true='false' vs model='true'
  Row 89: true='false' vs model='true'
  Row 91: true='false' vs model='true'
  Row 94: true='true' vs model='false'
  Row 98: true='true' vs model='fals

## Forecasting

In [8]:
"""
Evaluation Script: MSE for Forecasting Task
=============================================
Computes Mean Squared Error between the true answer and model response
for time series forecasting results.

Input:  results_forecasting.csv
        - Column: QA_list     (contains the true answer list/value after "answer" key)
        - Column: model_response (contains the model's predicted list/value)

Output: MSE statistics printed to console.
"""

import csv
import re
import numpy as np


def extract_list_from_text(text):
    """Extract a list of floats from text containing a bracketed list like [1.0, 2.0, ...]."""
    match = re.search(r'\[([^\]]+)\]', text)
    if not match:
        return None
    items = match.group(1).split(',')
    result = []
    for item in items:
        item = item.strip().strip("'\"") # remove whitespace and '"
        try:
            result.append(float(item)) # transform into float
        except ValueError:
            result.append(None)
    return [x for x in result if x is not None] # remove the None value from list


def extract_single_number(text):
    """
    Fallback: extract a single numeric value from free-text responses.
    Used when the answer/response is a single predicted value rather than a list.
    """
    nums = re.findall(r'[-+]?\d*\.?\d+', text) # find numeric value
    if nums:
        return [float(nums[-1])] # extract and return the last numeric value
    return None


def parse_answer(qa_text):
    """Extract the answer from the QA_list column."""
    if '"answer"' not in qa_text:
        return None

    answer_part = qa_text.split('"answer"')[-1] # take the last part {"question": "...", "answer": "[1, 2, 3]"})

    # Try list extraction first
    answer_list = extract_list_from_text(answer_part)
    if answer_list and len(answer_list) > 0:
        return answer_list

    # Fallback: single number from the answer text
    ans_match = re.search(r'"answer".*?:\s*"(.*?)"', qa_text, re.DOTALL) # extract content after answer within ""
    if ans_match:
        return extract_single_number(ans_match.group(1))

    return None


def parse_model_response(model_text):
    """Extract the predicted values from the model_response column."""
    # Try list extraction first
    model_list = extract_list_from_text(model_text)
    if model_list and len(model_list) > 0:
        return model_list

    # Fallback: single number
    return extract_single_number(model_text)


def compute_mse_forecasting(csv_path):
    """
    Parse the forecasting CSV and compute MSE between answer and model_response.

    For each row:
      1. Extract the answer (list or single value) from the QA_list column.
      2. Extract the prediction (list or single value) from the model_response column.
      3. Align by index using min(len(answer), len(model)) to handle length mismatches.
      4. Compute squared errors for each aligned pair.

    Returns a dict with detailed statistics.
    """
    with open(csv_path, 'r') as f:
        reader = csv.reader(f)
        header = next(reader)

        all_squared_errors = []
        row_mses = []
        row_details = []
        skipped_rows = []
        row_count = 0

        for row in reader:
            qa_text = row[2]       # QA_list column
            model_text = row[7]    # model_response column

            # --- Extract answer and model response ---
            answer_list = parse_answer(qa_text)
            model_list = parse_model_response(model_text)

            # --- Validate ---
            if answer_list is None or model_list is None:
                skipped_rows.append((row_count, "Could not parse list(s)"))
                row_count += 1
                continue

            if len(answer_list) == 0 or len(model_list) == 0:
                skipped_rows.append((row_count, "Empty list after parsing"))
                row_count += 1
                continue

            # --- Compute squared errors (aligned by position) ---
            min_len = min(len(answer_list), len(model_list))
            se = [(answer_list[i] - model_list[i]) ** 2 for i in range(min_len)]
            row_mse = np.mean(se)

            row_mses.append(row_mse)
            all_squared_errors.extend(se)
            row_details.append((row_count, len(answer_list), len(model_list), row_mse))

            row_count += 1

    # --- Aggregate statistics ---
    results = {
        "total_rows": row_count,
        "rows_used": len(row_mses),
        "rows_skipped": len(skipped_rows),
        "total_value_pairs": len(all_squared_errors),
        "mse_all_pairs": float(np.mean(all_squared_errors)) if all_squared_errors else None,
        "mse_avg_per_row": float(np.mean(row_mses)) if row_mses else None,
        "row_details": row_details,
        "skipped_rows": skipped_rows,
    }

    # Matched-length subset
    matched = [d for d in row_details if d[1] == d[2]]
    results["rows_matched_length"] = len(matched)
    results["mse_matched_length"] = float(np.mean([d[3] for d in matched])) if matched else None

    return results


def print_report(results):
    """Pretty-print the evaluation report."""
    print("=" * 60)
    print("  FORECASTING MSE EVALUATION REPORT")
    print("=" * 60)
    print(f"  Total rows in CSV:          {results['total_rows']}")
    print(f"  Rows successfully parsed:   {results['rows_used']}")
    print(f"  Rows skipped (unparseable): {results['rows_skipped']}")
    print(f"  Total value pairs compared: {results['total_value_pairs']}")
    print("-" * 60)
    print(f"  MSE (all value pairs):      {results['mse_all_pairs']:.6f}")
    print(f"  MSE (avg of per-row MSEs):  {results['mse_avg_per_row']:.6f}")
    print("-" * 60)
    print(f"  Rows with matching lengths: {results['rows_matched_length']}/{results['rows_used']}")
    if results['mse_matched_length'] is not None:
        print(f"  MSE (matched-length only):  {results['mse_matched_length']:.6f}")
    print("=" * 60)

    # Top 10 rows by MSE
    sorted_rows = sorted(results['row_details'], key=lambda x: x[3], reverse=True)
    print("\n  Top 10 rows by per-row MSE:")
    print(f"  {'Row':<6} {'Ans Len':<10} {'Model Len':<10} {'Row MSE':>15}")
    print("  " + "-" * 45)
    for r, alen, mlen, mse in sorted_rows[:10]:
        flag = " *" if alen != mlen else ""
        print(f"  {r:<6} {alen:<10} {mlen:<10} {mse:>15.4f}{flag}")
    print("  (* = length mismatch)")

    # Skipped rows
    if results['skipped_rows']:
        print(f"\n  Skipped rows:")
        for idx, reason in results['skipped_rows']:
            print(f"    Row {idx}: {reason}")


if __name__ == "__main__":
    CSV_PATH = "/content/drive/My Drive/CADE/results/deepseekv3.2/results_forecasting.csv"
    results = compute_mse_forecasting(CSV_PATH)
    print_report(results)

  FORECASTING MSE EVALUATION REPORT
  Total rows in CSV:          379
  Rows successfully parsed:   379
  Rows skipped (unparseable): 0
  Total value pairs compared: 7305
------------------------------------------------------------
  MSE (all value pairs):      377504.950932
  MSE (avg of per-row MSEs):  270979.443942
------------------------------------------------------------
  Rows with matching lengths: 373/379
  MSE (matched-length only):  275338.351697

  Top 10 rows by per-row MSE:
  Row    Ans Len    Model Len          Row MSE
  ---------------------------------------------
  43     28         28           30448874.3214
  231    30         30           25025738.8333
  291    32         32            9520624.8438
  377    12         12            7105799.4167
  18     25         25            6606866.4800
  178    28         28            6056280.0357
  94     26         26            4051880.2692
  76     31         31            2067086.1293
  40     30         30            1

## Imputation

In [9]:
"""
Evaluation Script: MSE for Imputation Task
============================================
Computes Mean Squared Error between the true answer and model response
for time series imputation results.

Input:  results_imputation.csv
        - Column: QA_list     (contains the true answer list after "answer" key)
        - Column: model_response (contains the model's predicted list)

Output: MSE statistics printed to console.
"""

import csv
import re
import numpy as np


def extract_list_from_text(text):
    """Extract a list of floats from text containing a bracketed list like [1.0, 2.0, ...]."""
    match = re.search(r'\[([^\]]+)\]', text)
    if not match:
        return None
    items = match.group(1).split(',')
    result = []
    for item in items:
        item = item.strip().strip("'\"")
        try:
            result.append(float(item))
        except ValueError:
            result.append(None)  # e.g. 'X' or non-numeric tokens
    return [x for x in result if x is not None]


def compute_mse_imputation(csv_path):
    """
    Parse the imputation CSV and compute MSE between answer and model_response.

    For each row:
      1. Extract the answer list from the QA_list column (after the "answer" key).
      2. Extract the predicted list from the model_response column.
      3. Align by index using min(len(answer), len(model)) to handle length mismatches.
      4. Compute squared errors for each aligned pair.

    Returns a dict with detailed statistics.
    """
    with open(csv_path, 'r') as f:
        reader = csv.reader(f)
        header = next(reader)

        all_squared_errors = []   # every individual (answer_i - model_i)^2
        row_mses = []             # per-row MSE values
        row_details = []          # (row_index, answer_len, model_len, row_mse)
        skipped_rows = []         # rows that couldn't be parsed
        row_count = 0

        for row in reader:
            qa_text = row[2]       # QA_list column
            model_text = row[7]    # model_response column

            # --- Extract answer list ---
            answer_list = None
            if '"answer"' in qa_text:
                answer_part = qa_text.split('"answer"')[-1]
                answer_list = extract_list_from_text(answer_part)

            # --- Extract model response list ---
            model_list = extract_list_from_text(model_text)

            # --- Validate ---
            if answer_list is None or model_list is None:
                skipped_rows.append((row_count, "Could not parse list(s)"))
                row_count += 1
                continue

            if len(answer_list) == 0 or len(model_list) == 0:
                skipped_rows.append((row_count, "Empty list after parsing"))
                row_count += 1
                continue

            # --- Compute squared errors (aligned by position) ---
            min_len = min(len(answer_list), len(model_list))
            se = [(answer_list[i] - model_list[i]) ** 2 for i in range(min_len)]
            row_mse = np.mean(se)

            row_mses.append(row_mse)
            all_squared_errors.extend(se)
            row_details.append((row_count, len(answer_list), len(model_list), row_mse))

            row_count += 1

    # --- Aggregate statistics ---
    results = {
        "total_rows": row_count,
        "rows_used": len(row_mses),
        "rows_skipped": len(skipped_rows),
        "total_value_pairs": len(all_squared_errors),
        "mse_all_pairs": float(np.mean(all_squared_errors)) if all_squared_errors else None,
        "mse_avg_per_row": float(np.mean(row_mses)) if row_mses else None,
        "row_details": row_details,
        "skipped_rows": skipped_rows,
    }

    # Matched-length subset
    matched = [d for d in row_details if d[1] == d[2]]
    results["rows_matched_length"] = len(matched)
    results["mse_matched_length"] = float(np.mean([d[3] for d in matched])) if matched else None

    return results


def print_report(results):
    """Pretty-print the evaluation report."""
    print("=" * 60)
    print("  IMPUTATION MSE EVALUATION REPORT")
    print("=" * 60)
    print(f"  Total rows in CSV:          {results['total_rows']}")
    print(f"  Rows successfully parsed:   {results['rows_used']}")
    print(f"  Rows skipped (unparseable): {results['rows_skipped']}")
    print(f"  Total value pairs compared: {results['total_value_pairs']}")
    print("-" * 60)
    print(f"  MSE (all value pairs):      {results['mse_all_pairs']:.6f}")
    print(f"  MSE (avg of per-row MSEs):  {results['mse_avg_per_row']:.6f}")
    print("-" * 60)
    print(f"  Rows with matching lengths: {results['rows_matched_length']}/{results['rows_used']}")
    if results['mse_matched_length'] is not None:
        print(f"  MSE (matched-length only):  {results['mse_matched_length']:.6f}")
    print("=" * 60)

    # Top 10 rows by MSE
    sorted_rows = sorted(results['row_details'], key=lambda x: x[3], reverse=True)
    print("\n  Top 10 rows by per-row MSE:")
    print(f"  {'Row':<6} {'Ans Len':<10} {'Model Len':<10} {'Row MSE':>15}")
    print("  " + "-" * 45)
    for r, alen, mlen, mse in sorted_rows[:10]:
        flag = " *" if alen != mlen else ""
        print(f"  {r:<6} {alen:<10} {mlen:<10} {mse:>15.4f}{flag}")
    print("  (* = length mismatch)")

    # Skipped rows
    if results['skipped_rows']:
        print(f"\n  Skipped rows:")
        for idx, reason in results['skipped_rows']:
            print(f"    Row {idx}: {reason}")


if __name__ == "__main__":
    CSV_PATH = "/content/drive/My Drive/CADE/results/deepseekv3.2/results_imputation.csv"
    results = compute_mse_imputation(CSV_PATH)
    print_report(results)

  IMPUTATION MSE EVALUATION REPORT
  Total rows in CSV:          400
  Rows successfully parsed:   400
  Rows skipped (unparseable): 0
  Total value pairs compared: 69171
------------------------------------------------------------
  MSE (all value pairs):      58084.063221
  MSE (avg of per-row MSEs):  87334.542350
------------------------------------------------------------
  Rows with matching lengths: 364/400
  MSE (matched-length only):  95552.204898

  Top 10 rows by per-row MSE:
  Row    Ans Len    Model Len          Row MSE
  ---------------------------------------------
  8      113        113          32238282.3053
  39     130        130           1196300.7115
  56     105        105            403178.5214
  73     200        200            113626.8200
  357    160        160            100315.6109
  187    178        178             91804.4101
  368    179        179             82342.1899
  317    182        182             75525.0673
  12     142        142             74